[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/31_gradient_accumulation_solution.ipynb)

# 🟢 Solution: Gradient Accumulation

*Training · Easy*

Reference implementation. Try it yourself in `31_gradient_accumulation.ipynb` first.

---
Implement **gradient accumulation**: run several small micro-batches and apply
a single optimizer step, so the update matches what one large batch would have
produced.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches):
    ...  # -> total_loss (a float)
```

- `model`: an `nnx.Module`
- `optimizer`: an `nnx.Optimizer`
- `loss_fn`: `loss_fn(predictions, targets) -> scalar`
- `micro_batches`: a list of `(x, y)` pairs

### Rules
- Scale each micro-batch loss by `1 / len(micro_batches)` **before** taking
  gradients
- Accumulate the gradient pytrees, then call `optimizer.update` **once**
- Return the summed (already-scaled) loss as a Python float

### Why divide by n
Gradients are linear in the loss, so
$\nabla(\tfrac{1}{n}\sum_i L_i) = \tfrac{1}{n}\sum_i \nabla L_i$. Dividing each
micro-loss by `n` and summing the gradients reproduces the mean-reduced
full-batch gradient exactly.

Skip the division and your effective learning rate is multiplied by `n` — the
model still trains, often looks fine for a while, then diverges. That silent
scaling is the classic bug.

### The subtlety worth knowing
This is exact **only when the micro-batches are the same size**. `loss_fn`
typically returns a *mean* over its micro-batch, and an unweighted mean of
means is not the mean of the whole unless every group has equal weight. With a
ragged last batch you must weight each micro-batch by its row count and divide
by the total. Interviewers like this one because the code looks correct either
way.

### ⚠️ What is different from PyTorch
There is **no `zero_grad`**. PyTorch accumulates into `p.grad` as a side effect,
so you must clear it first — and forgetting is a classic bug. `nnx.grad` returns
a fresh gradient tree on every call, so accumulation is something you do
explicitly with `jax.tree.map`, and the bug cannot be written.

`nnx.Optimizer` does mutate the model in place, which is the one stateful
concession NNX makes for ergonomics — underneath it is still a functional
update applied to the module's parameter State.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


def accumulated_step(model, optimizer, loss_fn, micro_batches):
    n = len(micro_batches)
    total_loss = 0.0
    acc = None

    # No optimizer.zero_grad() — nnx.grad hands back a fresh tree each call,
    # so nothing accumulates behind our back.
    for x, y in micro_batches:
        loss, grads = nnx.value_and_grad(
            lambda m: loss_fn(m(x), y) / n
        )(model)

        # Sum the gradient trees; this is the full-batch gradient.
        acc = grads if acc is None else jax.tree.map(jnp.add, acc, grads)
        total_loss += float(loss)

    optimizer.update(model, acc)
    return total_loss

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
import optax
from flax import nnx

model = nnx.Linear(4, 1, rngs=nnx.Rngs(params=0))
opt = nnx.Optimizer(model, optax.sgd(0.1), wrt=nnx.Param)

x = jax.random.normal(jax.random.key(1), (8, 4))
y = jax.random.normal(jax.random.key(2), (8,))
mse = lambda p, t: jnp.mean((p.squeeze(-1) - t) ** 2)

micro = [(x[:4], y[:4]), (x[4:], y[4:])]
print("loss:", accumulated_step(model, opt, mse, micro))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gradient_accumulation")